# Data Preparation

## Pilot Study Data Preparation

In [30]:
import os
import random
from pydub import AudioSegment
import json

# Define paths relative to the notebook workspace
DATA_DIR = "/teamspace/studios/turkish-music/anatolian-SAM/data/synthesized"
MIXED_DIR = "../data/mixed"


# with open(os.path.join("..", "data", "soundfonts_programs_tr.json"), "r") as f:
#     soundfont_programs = json.load(f)

# INSTRUMENTS = [sf["name"] for sf in soundfont_programs["soundfonts"]]
# INSTRUMENT_SYNONYMS = {
#     "baglama_saz": ["bağlama", "saz", "baglama", "turkish saz", "bağlama saz"],
#     "kanun": ["kanun", "qanun", "turkish zither"],
#     "kemence": ["kemençe", "kemence", "black sea fiddle", "karadeniz kemençesi"],
#     "ney": ["ney", "turkish flute", "reed flute"],
#     "ud": ["ud", "oud", "turkish lute"],
#     "zurna": ["zurna", "turkish oboe"],
# }


INSTRUMENT_SYNONYMS = {
    "baglama_saz": ["bağlama"],
    "zurna": ["zurna"],
}

INSTRUMENTS = ["baglama_saz", "zurna"]

os.makedirs(MIXED_DIR, exist_ok=True)

print("Directories setup complete.")

Directories setup complete.


In [14]:
SR = 22050  # Standard sample rate
CHUNK_LENGTH = 5  # seconds
SAMPLES_PER_CHUNK = SR * CHUNK_LENGTH

use this probability distribution:

70% of your dataset: Mix exactly 2 random instruments. (This builds your strong foundational weights).

25% of your dataset: Mix exactly 3 random instruments. (This teaches the model to handle some complexity).

5% of your dataset: Mix 4 random instruments. (This makes the model robust for real-world YouTube recordings).

0%: Do not bother mixing 5 or 6 simultaneously for this pilot study.

In [15]:
def get_random_5s_chunk(audio_segment, chunk_length_ms=5000):
    """
    Extracts a random 5-second slice from a full audio file.
    Pads with silence if the original file is too short.
    """
    audio_length = len(audio_segment)  # pydub measures length in milliseconds

    # Edge Case: The song is shorter than 5 seconds
    if audio_length <= chunk_length_ms:
        # Pad the end with pure silence to guarantee a perfect 5.000s tensor
        padding = AudioSegment.silent(duration=(chunk_length_ms - audio_length))
        return audio_segment + padding

    # Standard Case: The song is longer than 5 seconds
    # Pick a random start time that leaves exactly enough room for the 5s chunk
    max_start_time = audio_length - chunk_length_ms
    start_time = random.randint(0, max_start_time)
    end_time = start_time + chunk_length_ms

    # Slice the audio exactly like a Python list
    return audio_segment[start_time:end_time], start_time, end_time

In [31]:
def generate_training_tuple(tuple_id, instruments, synonyms=None):
    """
    Generates a single training tuple (Mixture, Target, Prompt) following
    the strict curriculum learning probability distribution.
    """

    # 1. Determine Polyphony (Number of Stems)
    num_instruments = random.choices([2, 3, 4], weights=[0.70, 0.25, 0.05], k=1)[0]

    # 2. Build the Acoustic Foundation
    # Guarantee at least two contrasting classes are present so separation actually happens
    base_mix = random.sample(instruments, 2)

    # 3. Add the Repetitions (The Chorus Effect)
    remaining_stems = num_instruments - 2
    if remaining_stems > 0:
        # random.choices allows replacement (duplicates)
        extras = random.choices(instruments, k=remaining_stems)
        selected_instruments = base_mix + extras
    else:
        selected_instruments = base_mix

    # Optional: Shuffle so the structure isn't predictable in your logs
    random.shuffle(selected_instruments)

    # 4. Choose the Ground Truth Target
    # We use set() to remove duplicates before picking.
    # If selected_instruments is ["baglama", "baglama", "zurna"],
    # set() reduces it to ["baglama", "zurna"] so both classes have an equal 50/50 chance of being the text prompt.
    target_instrument = random.choice(list(set(selected_instruments)))

    mixed_audio = None
    target_audio = None

    selected_instrument_info = {}

    # We need counters for both target and non-target stems to avoid overwriting JSON keys
    target_idx = 0
    non_target_idx = 0

    for inst in selected_instruments:
        inst_dir = os.path.join(DATA_DIR, inst)
        available_files = [f for f in os.listdir(inst_dir) if f.endswith(".wav")]

        # ASYNCHRONOUS MIXING: Pick a random file to prevent the 'Perfect Unison' trap
        random_file = random.choice(available_files)
        file_path = os.path.join(inst_dir, random_file)

        # Load the clean audio stem
        full_audio = AudioSegment.from_wav(file_path)

        # crop to 5-second stem
        stem, start_time, end_time = get_random_5s_chunk(full_audio)

        # 1. Overlay the complex mixture (Happens for ALL instruments)
        if mixed_audio is None:
            mixed_audio = stem
        else:
            mixed_audio = mixed_audio.overlay(stem)

        inst_metadata = {
            "instrument": inst,
            "song": random_file,
            "start_time": start_time,
            "end_time": end_time,
        }

        # 2. Overlay the Target Audio (Happens ONLY for the target instrument)
        if inst == target_instrument:
            if target_audio is None:
                target_audio = stem
            else:
                # MIX the identical instruments together instead of overwriting!
                target_audio = target_audio.overlay(stem)

            # Save metadata dynamically (e.g., "target_0", "target_1")
            selected_instrument_info[f"target_{target_idx}"] = inst_metadata
            target_idx += 1

        # 3. Handle the Non-Target (Noise) Metadata
        else:
            selected_instrument_info[f"noise_{non_target_idx}"] = inst_metadata
            non_target_idx += 1

    # 4. Save the generated .wav files
    mixture_path = os.path.join(
        MIXED_DIR, f"mix_{tuple_id:04d}_{num_instruments}inst.wav"
    )
    target_path = os.path.join(
        MIXED_DIR, f"target_{tuple_id:04d}_{target_instrument}.wav"
    )

    mixed_audio.export(mixture_path, format="wav")
    target_audio.export(target_path, format="wav")

    # 5. Format the Text Prompt for the SAM Model
    if synonyms:
        prompt = random.choice(synonyms[target_instrument])
    else:
        prompt = target_instrument

    return {
        "prompt": prompt,
        "mixture_path": mixture_path,
        "target_path": target_path,
        "selected_instrument_info": selected_instrument_info,
    }

In [26]:
from tqdm.autonotebook import tqdm
from torch.utils.data import random_split
import torch


def build_dataset(
    instruments, type="tr", num_samples=720, test_size=0.2, synonyms=None
):
    """
    Executes the generation loop to build the pilot dataset.
    720 samples equals roughly 1 hour of audio (if chunks are 5 seconds).
    """
    train_samples = int(num_samples * (1 - test_size))
    test_samples = num_samples - train_samples

    print(f"Initializing generation of {num_samples} training tuples...")
    print(f" - {train_samples} samples for training")
    print(f" - {test_samples} samples for validation")
    dataset_metadata = []

    train_subset, val_subset = random_split(
        list(range(num_samples)),
        [train_samples, test_samples],
        generator=torch.Generator().manual_seed(42),
    )

    for i in tqdm(range(num_samples), desc="Generating tuples"):
        metadata = generate_training_tuple(i, instruments, synonyms=synonyms)
        dataset_metadata.append(metadata)

    # Save the mapping file for the fine-tuning loop
    jsonl_path = os.path.join("..", "data", f"mixed_metadata_{type}.jsonl")
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for entry in dataset_metadata:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    # save the training and validation splits as well
    with open(f"../data/train_metadata_{type}.jsonl", "w", encoding="utf-8") as f:
        for idx in train_subset.indices:
            f.write(json.dumps(dataset_metadata[idx], ensure_ascii=False) + "\n")

    with open(f"../data/val_metadata_{type}.jsonl", "w", encoding="utf-8") as f:
        for idx in val_subset.indices:
            f.write(json.dumps(dataset_metadata[idx], ensure_ascii=False) + "\n")

    print(f"Dataset complete! Metadata mapped and saved to {jsonl_path}")
    print(
        f"Train and validation splits saved to ../data/train_metadata_{type}.jsonl and ../data/val_metadata_{type}.jsonl respectively."
    )

In [ ]:
# generate an initial pilot batch of 720 tuples
build_dataset(INSTRUMENTS, num_samples=720, synonyms=INSTRUMENT_SYNONYMS)

Initializing generation of 720 training tuples...
 - 576 samples for training
 - 144 samples for validation


Generating tuples: 100%|██████████| 720/720 [05:44<00:00,  2.09it/s]

Dataset complete! Metadata mapped and saved to ../data/mixed_metadata_tr.jsonl
Train and validation splits saved to ../data/train_metadata_tr.jsonl and ../data/val_metadata_tr.jsonl respectively.


## Data Preparation for The Ablation Study

Prepare the data for general (western) synthesized instruments, using the same distribution as above. This will be used to test the model's performance on more familiar sounds and to compare against the Anatolian instruments.

In [32]:
# with open(os.path.join("..", "data", "soundfonts_programs_ww.json"), "r") as f:
#     soundfont_programs = json.load(f)

# INSTRUMENTS = [sf["name"] for sf in soundfont_programs["soundfonts"]]
INSTRUMENTS = ["steel_string_guitar", "bagpipes"]
INSTRUMENTS_SYNONYMS = {
    "steel_string_guitar": ["acoustic guitar"],
    "bagpipes": ["bagpipes"],
}

In [33]:
MIXED_DIR = "../data/mixed/ww"
os.makedirs(MIXED_DIR, exist_ok=True)

In [34]:
build_dataset(INSTRUMENTS, type="ww", num_samples=720, synonyms=INSTRUMENTS_SYNONYMS)

Initializing generation of 720 training tuples...
 - 576 samples for training
 - 144 samples for validation


Generating tuples: 100%|██████████| 720/720 [14:32<00:00,  1.21s/it]

Dataset complete! Metadata mapped and saved to ../data/mixed_metadata_ww.jsonl
Train and validation splits saved to ../data/train_metadata_ww.jsonl and ../data/val_metadata_ww.jsonl respectively.
